# Module 17: Capstone Project

## End-to-End Rainfall Visualization Dashboard

**Goal**: Build a complete, publication-quality 4-panel dashboard analyzing CHIRPS rainfall data for Ethiopia. This capstone integrates everything you have learned:

- Loading and exploring NetCDF data with xarray
- Computing statistical summaries (mean, std, trends)
- Geographic mapping with cartopy
- Time series analysis with pandas and scipy
- Seasonal cycle and climatology computation
- Regional comparison and aggregation
- Publication-quality export

**Data**: CHIRPS v2.0 monthly rainfall (mm/month), 0.05 deg, 1981-2022.

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
from matplotlib import rcParams
from scipy import stats
from scipy.ndimage import gaussian_filter
import warnings; warnings.filterwarnings('ignore')

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False
    print('Cartopy not installed — using simple imshow for maps')

# Load CHIRPS
ds = xr.open_dataset('../chirps_1981_2022.nc', engine='netcdf4')
precip = ds['precip']
print(ds)

## 2. Data Exploration

In [ ]:
# Basic statistics
print('=== Dataset Info ===')
print(f'Time steps: {precip.shape[0]}')
print(f'Lat x Lon: {precip.shape[1]} x {precip.shape[2]}')
print(f'Time range: {str(precip.time.values[0])[:7]} to {str(precip.time.values[-1])[:7]}')
print(f'Global mean: {precip.mean().values:.2f} mm/month')
print(f'Global std:  {precip.std().values:.2f} mm/month')

# Ethiopia subset
eth = precip.sel(lat=slice(15.0, 3.0), lon=slice(33.0, 48.0))
print(f'\nEthiopia subset shape: {eth.shape}')
print(f'Ethiopia mean: {eth.mean().values:.2f} mm/month')

In [ ]:
# Compute key statistics arrays
# Long-term monthly climatology (mean for each month across all years)
monthly_clim = eth.groupby('time.month').mean(dim='time')

# Long-term mean annual rainfall
mean_annual_map = eth.mean(dim='time') * 12  # convert monthly mean to annual

# Standard deviation map
std_map = eth.std(dim='time')

# Ethiopia-wide time series
eth_ts = eth.mean(dim=['lat', 'lon'])
rainfall_series = eth_ts.to_series()
annual = rainfall_series.resample('YE').sum()
annual.index = annual.index.year

print('Statistics computed.')
print(f'Ethiopia annual mean: {annual.mean():.1f} mm/yr')

## 3. Panel 1: Ethiopia Rainfall Map

A geographic map of mean annual precipitation over Ethiopia.

In [ ]:
def create_rainfall_map(ax, data, title, cmap='viridis', cbar_label='mm/year'):
    """Create a geographic rainfall map on the given axes."""
    lats = eth.lat.values
    lons = eth.lon.values
    
    if HAS_CARTOPY:
        ax = plt.subplot(ax, projection=ccrs.PlateCarree())
        ax.add_feature(cfeature.COASTLINE, lw=0.5)
        ax.add_feature(cfeature.BORDERS, lw=0.5, alpha=0.5)
        ax.add_feature(cfeature.LAKES, alpha=0.3, color='lightblue')
        ax.add_feature(cfeature.RIVERS, lw=0.3, alpha=0.4)
        
        im = ax.pcolormesh(lons, lats, data, cmap=cmap, 
                           transform=ccrs.PlateCarree(), shading='auto')
        ax.set_extent([33, 48, 3, 15], crs=ccrs.PlateCarree())
        gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5,
                          linestyle='--', color='gray')
        gl.top_labels = False
        gl.right_labels = False
        gl.xlabel_style = {'size': 7}
        gl.ylabel_style = {'size': 7}
    else:
        im = ax.imshow(data, extent=[33, 48, 3, 15], origin='upper',
                       cmap=cmap, aspect='auto')
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        ax.grid(True, alpha=0.2)
    
    ax.set_title(title, fontsize=10, fontweight='bold')
    return im

In [ ]:
# Build Panel 1
fig = plt.figure(figsize=(5, 6))
if HAS_CARTOPY:
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
else:
    ax = fig.add_subplot(1, 1, 1)

im = create_rainfall_map(ax, mean_annual_map.values,
                         'Mean Annual Precipitation (1981-2022)')
cbar = fig.colorbar(im, ax=[ax], shrink=0.85, pad=0.05)
cbar.set_label('mm/year', fontsize=9)
cbar.ax.tick_params(labelsize=7)
plt.tight_layout()
plt.show()

## 4. Panel 2: Time Series with Trend Line

Annual rainfall with linear trend and uncertainty band.

In [ ]:
# Compute trend
x = annual.index.values
y = annual.values
slope, intercept, r_val, p_val, std_err = stats.linregress(x, y)
trend = intercept + slope * x

print(f'Annual trend: {slope:.2f} mm/yr (p={p_val:.4f})')
print(f'Significant at 95%: {"Yes" if p_val < 0.05 else "No"}')

# 5-year moving average
rolling = pd.Series(y, index=x).rolling(window=5, center=True).mean()

fig, ax = plt.subplots(figsize=(8, 4))

# Bars
ax.bar(x, y, color='#4575b4', alpha=0.6, width=0.8, label='Annual total', zorder=1)

# Trend line
ax.plot(x, trend, color='#d73027', lw=2, label=f'Trend: {slope:.1f} mm/yr', zorder=3)

# Confidence band (95% CI)
t_val = stats.t.ppf(0.975, len(x) - 2)
ci = t_val * std_err * np.sqrt(1/len(x) + (x - x.mean())**2 / np.sum((x - x.mean())**2))
ax.fill_between(x, trend - ci, trend + ci, color='#d73027', alpha=0.1, label='95% CI', zorder=2)

# Rolling mean
ax.plot(x, rolling.values, color='#2c7bb6', lw=1.5, ls='--', label='5-yr moving avg', zorder=4)

# Reference line
ax.axhline(y.mean(), color='gray', ls=':', lw=0.8, alpha=0.7, label=f'Mean: {y.mean():.0f} mm')

ax.set_title('Annual Rainfall Time Series with Trend', fontsize=12, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Precipitation (mm/year)')
ax.legend(fontsize=8, framealpha=0.9, loc='upper left')
ax.grid(True, alpha=0.3)

# Inset with statistics
stats_text = (
    f'Slope: {slope:.2f} mm/yr\n'
    f'p-value: {p_val:.4f}\n'
    f'R²: {r_val**2:.3f}\n'
    f'n = {len(x)} years'
)
ax.text(0.98, 0.05, stats_text, transform=ax.transAxes,
        fontsize=8, verticalalignment='bottom', horizontalalignment='right',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

## 5. Panel 3: Seasonal Cycle (Monthly Climatology)

Mean monthly rainfall with variability bands.

In [ ]:
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# Monthly statistics
monthly_mean = rainfall_series.groupby(rainfall_series.index.month).mean()
monthly_std = rainfall_series.groupby(rainfall_series.index.month).std()
monthly_min = rainfall_series.groupby(rainfall_series.index.month).min()
monthly_max = rainfall_series.groupby(rainfall_series.index.month).max()

# Add contrasting years
years_to_show = [1984, 2005, 2018]  # dry, normal, wet (example)
year_colors = {1984: '#d73027', 2005: '#4575b4', 2018: '#2c7bb6'}

fig, ax = plt.subplots(figsize=(8, 4.5))

# Climatology bar with error bars
ax.bar(range(1, 13), monthly_mean.values, yerr=monthly_std.values,
       color='steelblue', alpha=0.6, capsize=3,
       error_kw={'lw': 0.8, 'color': 'gray'},
       label='Mean ± 1 SD', width=0.7, zorder=2)

# Range (min-max)
ax.fill_between(range(1, 13), monthly_min.values, monthly_max.values,
                alpha=0.12, color='steelblue', label='Min-Max range', zorder=1)

# Individual years for comparison
for year, color in year_colors.items():
    year_data = rainfall_series[rainfall_series.index.year == year]
    ax.plot(range(1, 13), year_data.values, 'o-', color=color, lw=1.2,
            markersize=4, label=str(year), zorder=3)

ax.set_title('Monthly Rainfall Climatology (1981-2022)', fontsize=12, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Precipitation (mm/month)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names, fontsize=8)
ax.legend(fontsize=8, ncol=2, framealpha=0.9)
ax.grid(True, axis='y', alpha=0.3)

# Annotate rainy seasons
ax.annotate('Belg (small rains)', xy=(4, monthly_mean.loc[4]),
            xytext=(2, monthly_mean.loc[4] + 40),
            arrowprops=dict(arrowstyle='->', color='green', lw=1),
            fontsize=8, color='green', style='italic')
ax.annotate('Kiremt (main rains)', xy=(8, monthly_mean.loc[8]),
            xytext=(6.5, monthly_mean.loc[8] + 50),
            arrowprops=dict(arrowstyle='->', color='blue', lw=1),
            fontsize=8, color='blue', style='italic')

plt.tight_layout()
plt.show()

## 6. Panel 4: Regional Comparison Bar Chart

Compare rainfall regimes across Ethiopian regions.

In [ ]:
# Define regions with lat/lon bounds
regions = {
    'Tigray (N)':      {'lat': slice(15.0, 12.5), 'lon': slice(36.5, 40.0)},
    'Amhara (NW)':     {'lat': slice(12.5, 10.0), 'lon': slice(36.0, 40.0)},
    'Oromia (C)':      {'lat': slice(10.0, 7.0),  'lon': slice(36.0, 42.0)},
    'SNNP (SW)':       {'lat': slice(7.0, 4.5),   'lon': slice(35.0, 38.5)},
    'Somali (SE)':     {'lat': slice(9.0, 4.0),   'lon': slice(42.0, 48.0)},
    'Afar (E)':        {'lat': slice(14.0, 10.0), 'lon': slice(40.0, 42.5)},
}

region_stats = {}
for name, bounds in regions.items():
    subset = precip.sel(lat=bounds['lat'], lon=bounds['lon'])
    ts = subset.mean(dim=['lat', 'lon']).to_series()
    clim = ts.groupby(ts.index.month).mean()
    region_stats[name] = {
        'annual_mean': ts.mean() * 12,
        'annual_std': ts.std() * np.sqrt(12),
        'dry_month': clim.idxmin(),
        'wet_month': clim.idxmax(),
        'wet_season_frac': ts[ts.index.month.isin([6,7,8])].sum() / ts.sum(),
    }

df_regions = pd.DataFrame(region_stats).T
print(df_regions)

In [ ]:
# Create multi-panel regional comparison
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

region_names = list(regions.keys())
colors_regions = ['#1a9641', '#a6d96a', '#fdae61', '#f46d43', '#d73027', '#4575b4']

# Panel A: Annual mean
ax = axes[0, 0]
vals = df_regions['annual_mean'].values
errs = df_regions['annual_std'].values
bars = ax.barh(range(len(vals)), vals, xerr=errs, color=colors_regions,
               capsize=3, tick_label=region_names)
ax.set_title('A: Mean Annual Rainfall', fontsize=10, fontweight='bold')
ax.set_xlabel('mm/year')
ax.grid(True, axis='x', alpha=0.3)
for bar, val in zip(bars, vals):
    ax.text(val + 10, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}', va='center', fontsize=7)

# Panel B: Seasonal cycle for all regions
ax = axes[0, 1]
for i, (name, bounds) in enumerate(regions.items()):
    ts = precip.sel(lat=bounds['lat'], lon=bounds['lon']).mean(dim=['lat','lon']).to_series()
    clim = ts.groupby(ts.index.month).mean()
    ax.plot(range(1, 13), clim.values, color=colors_regions[i],
            label=name, marker='.', lw=1)
ax.set_title('B: Monthly Climatology by Region', fontsize=10, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('mm/month')
ax.set_xticks(range(1, 13))
ax.legend(fontsize=6, loc='upper right')
ax.grid(True, alpha=0.3)

# Panel C: Wet season proportion
ax = axes[1, 0]
vals = df_regions['wet_season_frac'].values * 100
ax.barh(range(len(vals)), vals, color=colors_regions,
        tick_label=region_names)
ax.set_title('C: JJA Fraction of Annual Total', fontsize=10, fontweight='bold')
ax.set_xlabel('% of annual rainfall')
ax.set_xlim(0, 100)
ax.grid(True, axis='x', alpha=0.3)
for i, val in enumerate(vals):
    ax.text(val + 1, i, f'{val:.0f}%', va='center', fontsize=7)

# Panel D: Coefficient of variation
ax = axes[1, 1]
cv = df_regions['annual_std'] / df_regions['annual_mean'] * 100
ax.barh(range(len(cv)), cv.values, color=colors_regions,
        tick_label=region_names)
ax.set_title('D: Coefficient of Variation', fontsize=10, fontweight='bold')
ax.set_xlabel('CV (%)')
ax.grid(True, axis='x', alpha=0.3)

plt.suptitle('Regional Comparison of Ethiopian Rainfall Climates',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. 4-Panel Dashboard (Combined)

Combine all four panels into a single publication-ready figure.

In [ ]:
# Create the complete dashboard
fig = plt.figure(figsize=(14, 10))

# GridSpec layout
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.2], height_ratios=[1, 1],
                       wspace=0.25, hspace=0.30,
                       left=0.06, right=0.95, bottom=0.07, top=0.93)

# Panel 1: Rainfall Map
if HAS_CARTOPY:
    ax1 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
else:
    ax1 = fig.add_subplot(gs[0, 0])
im = create_rainfall_map(ax1, mean_annual_map.values, 'A: Mean Annual Rainfall')

# Panel 2: Time Series
ax2 = fig.add_subplot(gs[0, 1])
ax2.bar(x, y, color='#4575b4', alpha=0.5, width=0.8, label='Annual')
ax2.plot(x, trend, color='#d73027', lw=2, label=f'Trend: {slope:.1f} mm/yr')
ax2.fill_between(x, trend - ci, trend + ci, color='#d73027', alpha=0.1, label='95% CI')
ax2.axhline(y.mean(), color='gray', ls=':', lw=0.8, alpha=0.5)
ax2.set_title('B: Annual Rainfall Trend', fontsize=10, fontweight='bold')
ax2.set_xlabel('Year')
ax2.set_ylabel('mm/year')
ax2.legend(fontsize=7, loc='upper left')
ax2.grid(True, alpha=0.3)

# Panel 3: Seasonal Cycle
ax3 = fig.add_subplot(gs[1, 0])
ax3.bar(range(1, 13), monthly_mean.values, yerr=monthly_std.values,
        color='steelblue', alpha=0.6, capsize=2, width=0.7)
ax3.fill_between(range(1, 13), monthly_min.values, monthly_max.values,
                alpha=0.12, color='steelblue')
ax3.set_title('C: Monthly Climatology', fontsize=10, fontweight='bold')
ax3.set_xlabel('Month')
ax3.set_ylabel('mm/month')
ax3.set_xticks(range(1, 13))
ax3.grid(True, axis='y', alpha=0.3)

# Panel 4: Regional Comparison
ax4 = fig.add_subplot(gs[1, 1])
vals = df_regions['annual_mean'].values
errs = df_regions['annual_std'].values
bars = ax4.barh(range(len(vals)), vals, xerr=errs, color=colors_regions,
               capsize=3, tick_label=['Tigray', 'Amhara', 'Oromia',
                                      'SNNP', 'Somali', 'Afar'])
ax4.set_title('D: Regional Comparison', fontsize=10, fontweight='bold')
ax4.set_xlabel('mm/year')
ax4.grid(True, axis='x', alpha=0.3)

# Add colorbar for the map
cbar_ax = fig.add_axes([0.06, 0.93, 0.40, 0.015])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
cbar.set_label('Mean Annual Precipitation (mm/year)', fontsize=8)
cbar.ax.tick_params(labelsize=7)

fig.suptitle('Ethiopia Rainfall Dashboard — CHIRPS v2.0 (1981-2022)',
             fontsize=14, fontweight='bold', y=0.98)

plt.show()

## 8. Export as Publication-Quality Images

Save the dashboard and individual panels for publication.

In [ ]:
import os
os.makedirs('exports', exist_ok=True)

# Export the combined dashboard
fig.savefig('exports/ethiopia_rainfall_dashboard.png', dpi=300, bbox_inches='tight')
fig.savefig('exports/ethiopia_rainfall_dashboard.pdf', dpi=300, bbox_inches='tight')
fig.savefig('exports/ethiopia_rainfall_dashboard.svg', dpi=300, bbox_inches='tight')

# Export individual panels
# Re-create and save Panel 1 alone (map)
fig1, ax1 = plt.subplots(figsize=(4, 4.5))
if HAS_CARTOPY:
    ax1 = fig1.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
im1 = create_rainfall_map(ax1, mean_annual_map.values, 'Mean Annual Rainfall')
fig1.colorbar(im1, ax=[ax1], shrink=0.85, label='mm/year')
fig1.savefig('exports/panel1_rainfall_map.png', dpi=300, bbox_inches='tight')
fig1.savefig('exports/panel1_rainfall_map.pdf', dpi=300, bbox_inches='tight')
plt.close(fig1)

print('All panels exported to exports/')
print('Files:')
for f in os.listdir('exports'):
    size_kb = os.path.getsize(f'exports/{f}') / 1024
    print(f'  {f}: {size_kb:.1f} KB')

## 9. Document the Process

### Methodology

| Step | Description | 
|------|-------------|
| 1. Data Loading | CHIRPS v2.0 monthly rainfall from `chirps_1981_2022.nc` via xarray |
| 2. Subsetting | Ethiopia bounding box: 3-15°N, 33-48°E |
| 3. Statistics | Mean, std, min, max, trend computed with numpy/scipy |
| 4. Panel 1 | Geographic map using Cartopy (PlateCarree projection) |
| 5. Panel 2 | Annual time series with OLS trend and 95% CI |
| 6. Panel 3 | Monthly climatology with variability bands |
| 7. Panel 4 | Regional bar chart with 6 Ethiopian regions |
| 8. Export | PNG (300 dpi), PDF, SVG for publication |

### Key Findings
- Ethiopia's mean annual rainfall (1981-2022): **{annual.mean():.0f} mm**
- Trend: **{slope:.2f} mm/year** (p={'significant' if p_val < 0.05 else 'not significant'})
- Main rainy season (Kiremt, JJA) dominates in northern and central regions
- Southeastern lowlands (Somali, Afar) are consistently drier with higher variability

### Tools Used
- **xarray** + **netCDF4** for NetCDF data handling
- **numpy** + **scipy** for statistical analysis
- **pandas** for time series resampling
- **matplotlib** for all plotting (OO interface)
- **cartopy** for geographic mapping

## 10. Exercises

1. **Interactive Dashboard**: Convert the 4-panel dashboard into an interactive widget using ipywidgets. Add a dropdown to select between viewing rainfall, anomaly, or CV maps.

2. **Seasonal Subsetting**: Add a slider that allows the user to select a specific year range. All four panels should update dynamically.

3. **Additional Panel**: Add a 5th panel showing the rainfall anomaly for the most recent year (2022) relative to the long-term mean.

4. **Trend Map**: Create a map showing the linear trend (slope) at each grid cell over the 42-year record. Highlight cells where the trend is statistically significant (p < 0.05).

5. **Export Automation**: Write a function that exports all panels at multiple resolutions (150, 300, 600 DPI) and in multiple formats (PNG, PDF, SVG, EPS) with a single call.

6. **Hovmöller Diagram**: Create a Hovmöller (time-longitude) diagram for a latitudinal band across Ethiopia, showing the seasonal progression of rainfall.

### Mini-Project

**Interactive Ethiopia Rainfall Explorer**: Build a complete interactive dashboard using ipywidgets that allows the user to:
- Select a year range (slider)
- Select a month or season (dropdown)
- Choose between mean, anomaly, or CV display (radio buttons)
- All panels update in real-time

Bonus: Add a click-to-plot feature where clicking on the map shows that location's time series in a separate panel.